In [31]:
from tkinter import *
from tkinter import ttk
from tkinter import messagebox
import mysql.connector

class Crud():
    def __init__(self):
        self.root = Tk()
        self.root.title("Crud Aluno")
        self.root.geometry(self.centralizar_janela(self.root, 650, 500))
        self.root.config(bg='black')

        # Variaveis
        self.var_id = StringVar()
        self.var_id.set('?')
        
      # Variaveis de imagem
        self.img_btn_add = PhotoImage(file=r"add.png")
        self.img_btn_update = PhotoImage(file=r"update.png")
        self.img_btn_delete = PhotoImage(file=r"delete.png")
        
        self.containers()
        self.itens_container01()
        self.itens_container02()
        self.preencher_tabela()
        
        self.root.mainloop()

    def containers(self):
        self.fr_container01 = Frame(self.root, height=200, width=650, bg='#3d4957')
        self.fr_container02 = Frame(self.root, height=300, width=650, bg='#eef2f5')
        self.fr_container01.propagate(0)
        self.fr_container02.propagate(0)
        self.fr_container01.pack()
        self.fr_container02.pack()

    def itens_container01(self):
        bg_color = self.fr_container01.cget('bg')
        self.fr_container_title = Frame(self.fr_container01, bg=bg_color)
        self.fr_container_infos_user = Frame(self.fr_container01, bg=bg_color)
        self.fr_container_buttons = Frame(self.fr_container01, bg=bg_color)

        self.lb_title = Label(self.fr_container_title, text='Sistema de cadastro de Aluno',
                              font='Calibri 19 bold', fg='#ffd546', bg=bg_color)
        
        # Entrys
        self.lb_id_aluno = Label(self.fr_container_infos_user, text="Id do Aluno", font='Calibri 11 bold', fg ="white", bg=bg_color)
        self.lb_id_aluno_values = Label(self.fr_container_infos_user, textvariable=self.var_id, font='Calibri 11 bold', fg ="white", bg=bg_color)

        self.en_nome_aluno = Entry(self.fr_container_infos_user, width=30, font="Calibri 11", bg=bg_color, fg='white', insertbackground='white')
        self.en_email_aluno = Entry(self.fr_container_infos_user, width=30, font='Calibri 11', bg=bg_color, fg='white', insertbackground='white')
        self.en_nome_curso = Entry(self.fr_container_infos_user, width=30, font='Calibri 11', bg=bg_color, fg='white', insertbackground='white')
        self.en_valor_curso = Entry(self.fr_container_infos_user, width=30, font='Calibri 11', bg=bg_color, fg='white', insertbackground='white')

        # Botoes
        self.btn_adicionar = Button(self.fr_container_buttons, image=self.img_btn_add, 
                            activebackground=bg_color, bg=bg_color, bd=0, 
                            cursor="hand2", command=self.adicionar_registro)

        self.btn_update = Button(self.fr_container_buttons, image=self.img_btn_update, 
                         activebackground=bg_color, bg=bg_color, bd=0, 
                         cursor="hand2", command=self.update_registro)

        self.btn_delete = Button(self.fr_container_buttons, image=self.img_btn_delete, 
                         activebackground=bg_color, bg=bg_color, bd=0, 
                         cursor="hand2", command=self.excluir_registro)
        
        # Layout
        self.fr_container_title.pack(anchor=W, padx=10)
        self.fr_container_infos_user.pack(anchor=W, padx=10)
        self.fr_container_buttons.pack(anchor=W, padx=215, pady=5)
        self.lb_title.pack()
        
        # Labels grid
        Label(self.fr_container_infos_user, text="Nome", fg="white", bg=bg_color).grid(row=1, column=0, sticky='w')
        self.en_nome_aluno.grid(row=1, column=1, sticky='w', pady=2)
        Label(self.fr_container_infos_user, text="Email", fg="white", bg=bg_color).grid(row=2, column=0, sticky='w')
        self.en_email_aluno.grid(row=2, column=1, sticky='w', pady=2)
        Label(self.fr_container_infos_user, text="Curso", fg="white", bg=bg_color).grid(row=3, column=0, sticky='w')
        self.en_nome_curso.grid(row=3, column=1, sticky='w', pady=2)
        Label(self.fr_container_infos_user, text="Valor", fg="white", bg=bg_color).grid(row=4, column=0, sticky='w')
        self.en_valor_curso.grid(row=4, column=1, sticky='w', pady=2)

        self.btn_adicionar.grid(row=0, column=0, padx=10)
        self.btn_update.grid(row=0, column=1, padx=10)
        self.btn_delete.grid(row=0, column=2, padx=10)

    def itens_container02(self):
        self.treeview = ttk.Treeview(self.fr_container02, columns=('id','nome','email','curso','valor'), show='headings') 
        for col in ('id','nome','email','curso','valor'):
            self.treeview.heading(col, text=col.capitalize())
        self.treeview.bind('<Double-1>', self.capturar_registros)
        self.treeview.pack(fill='both', expand=True, padx=10, pady=10)

    def conectar(self):
        return mysql.connector.connect(
            host='localhost', user='root', password='290118', database='crud_aluno'
        )

    def atualizar_tabela(self):
        try:
            conexao = self.conectar()
            cursor = conexao.cursor()
            cursor.execute('SELECT * FROM aluno')
            registros = cursor.fetchall()
            cursor.close()
            conexao.close()
            return registros
        except Exception as e:
            print(f'Erro ao conectar: {e}')
            return [] 

    def preencher_tabela(self):
        for i in self.treeview.get_children():
            self.treeview.delete(i)
        for registro in self.atualizar_tabela():
            self.treeview.insert("", 'end', values=registro)
            
    def validar_entrys(self):
        campos = [self.en_nome_aluno.get(), self.en_email_aluno.get(), self.en_nome_curso.get(), self.en_valor_curso.get()]
        return all(c.strip() != "" for c in campos)

    def adicionar_registro(self):
        if self.validar_entrys():
            try:
                conexao = self.conectar()
                cursor = conexao.cursor()
                query = "INSERT INTO aluno (nome, email, curso, valor) VALUES (%s, %s, %s, %s)"
                valores = (self.en_nome_aluno.get(), self.en_email_aluno.get(), self.en_nome_curso.get(), self.en_valor_curso.get())
                cursor.execute(query, valores)
                conexao.commit()
                
                messagebox.showinfo("Sucesso", "Aluno cadastrado com sucesso!")
                
                self.resetar_entreys() # Chamando o nome correto da função
                self.preencher_tabela()
                
                cursor.close()
                conexao.close()
            except Exception as e:
                messagebox.showerror("Erro", f"Erro no banco de dados: {e}")
        else:
            messagebox.showwarning("Aviso", "Preencha todos os campos!")

    def update_registro(self):
        id_aluno = self.var_id.get()
        if id_aluno != "?" and self.validar_entrys():
            if messagebox.askyesno("Confirmação", "Deseja atualizar?"):
                try:
                    conexao = self.conectar()
                    cursor = conexao.cursor()
                    query = "UPDATE aluno SET nome=%s, email=%s, curso=%s, valor=%s WHERE id=%s"
                    valores = (self.en_nome_aluno.get(), self.en_email_aluno.get(), self.en_nome_curso.get(), self.en_valor_curso.get(), id_aluno)
                    cursor.execute(query, valores)
                    conexao.commit()
                    self.resetar_entreys()
                    self.preencher_tabela()
                    conexao.close()
                except Exception as e:
                    messagebox.showerror("Erro", str(e))

    def excluir_registro(self):
        id_aluno = self.var_id.get()
        if id_aluno != "?":
            if messagebox.askyesno("Confirmação", "Excluir registro?"):
                try:
                    conexao = self.conectar()
                    cursor = conexao.cursor()
                    cursor.execute("DELETE FROM aluno WHERE id = %s", (id_aluno,))
                    conexao.commit()
                    self.resetar_entreys()
                    self.preencher_tabela()
                    conexao.close()
                except Exception as e:
                    messagebox.showerror("Erro", str(e))

    def resetar_entreys(self):
        self.en_nome_aluno.delete(0, END)
        self.en_email_aluno.delete(0, END)
        self.en_nome_curso.delete(0, END)
        self.en_valor_curso.delete(0, END)
        self.var_id.set("?")

    def capturar_registros(self, event):
        item = self.treeview.focus()
        valores = self.treeview.item(item, "values")
        if valores:
            self.resetar_entreys()
            self.var_id.set(valores[0])
            self.en_nome_aluno.insert(0, valores[1])
            self.en_email_aluno.insert(0, valores[2])
            self.en_nome_curso.insert(0, valores[3])
            self.en_valor_curso.insert(0, valores[4])

    def centralizar_janela(self, janela, largura, altura):
        largura_screen = janela.winfo_screenwidth()
        altura_screen = janela.winfo_screenheight()
        return f'{largura}x{altura}+{int(largura_screen/2 - largura/2)}+{int(altura_screen/2 - altura/2)}'

    
Crud()